# SparkClient: Advanced Spark Connect Options (KEP-107 Pattern)

This notebook demonstrates how to apply advanced Kubernetes configurations to interactive Spark Connect sessions using the KEP-107 options pattern.

### What you will learn:
1. Adding `Labels` and `Annotations` for cost attribution, monitoring, and metadata tracking.
2. Applying `NodeSelector` to route driver/executors to dedicated hardware pools (e.g., GPU nodes).
3. Specifying `Toleration` configurations for tainted nodes and spot instance execution.
4. Explicitly naming sessions with the `Name` option.
5. Combining `Driver`, `Executor`, and scheduling options into production deployment patterns.

## 1. Environment and Client Setup

Import classes from `kubeflow.spark` and initialize `SparkClient`.

In [ ]:
import os
import uuid

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import (
    Annotations,
    Driver,
    Executor,
    Labels,
    Name,
    NodeSelector,
    SparkClient,
    Toleration,
)

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)

print(f"SparkClient initialized for namespace: {namespace}")

## 2. Resource Organization with Labels and Annotations

Attach Kubernetes labels for filtering and cost accounting alongside annotations for runbooks, alerting, and operational metadata.

In [ ]:
session_name_1 = f"spark-annotated-{uuid.uuid4().hex[:8]}"

spark = client.connect(
    num_executors=1,
    resources_per_executor={"cpu": "1", "memory": "512m"},
    spark_conf={"spark.serializer": "org.apache.spark.serializer.KryoSerializer"},
    options=[
        Name(session_name_1),
        Labels(
            {
                "app": "spark",
                "team": "data-engineering",
                "environment": "production",
                "cost-center": "analytics",
            }
        ),
        Annotations(
            {
                "description": "Daily ETL pipeline for customer data",
                "owner": "data-team@company.com",
                "runbook": "https://wiki.company.com/runbooks/etl",
                "pagerduty": "spark-oncall",
            }
        ),
    ],
)

print(f"Spark session created with labels and annotations: {session_name_1}")
df = spark.range(100)
print(f"Processed {df.count()} rows")

spark.stop()
client.delete_session(name=session_name_1)
print(f"Session {session_name_1} stopped and cleaned up.")

## 3. Node Selection for Targeted Hardware

Direct pods to nodes matching specific labels, such as GPU pools or high-memory instances.

In [ ]:
session_name_2 = f"spark-gpu-{uuid.uuid4().hex[:8]}"

spark = client.connect(
    num_executors=1,
    resources_per_executor={
        "cpu": "1",
        "memory": "512m",
    },
    spark_conf={"spark.serializer": "org.apache.spark.serializer.KryoSerializer"},
    options=[
        Name(session_name_2),
        NodeSelector(
            {
                "kubernetes.io/os": "linux",
            }
        ),
    ],
)

print(f"Spark session scheduled with node selector: {session_name_2}")
df = spark.range(1000)
print(f"Processed {df.count()} rows on targeted executors")

spark.stop()
client.delete_session(name=session_name_2)
print(f"Session {session_name_2} stopped and cleaned up.")

## 4. Tolerations for Tainted Nodes and Spot Instances

Configure pod tolerations to permit scheduling on tainted nodes (such as dedicated Spark infrastructure or preemptible spot instances).

In [ ]:
session_name_3 = f"spark-spot-{uuid.uuid4().hex[:8]}"

spark = client.connect(
    num_executors=1,
    resources_per_executor={"cpu": "1", "memory": "512m"},
    spark_conf={"spark.serializer": "org.apache.spark.serializer.KryoSerializer"},
    options=[
        Name(session_name_3),
        Toleration(
            key="spark-workload",
            operator="Equal",
            value="true",
            effect="NoSchedule",
        ),
        Toleration(
            key="spot-instance",
            operator="Exists",
            effect="NoSchedule",
        ),
    ],
)

print(f"Spark session scheduled on tainted/spot nodes: {session_name_3}")
df = spark.range(10000)
print(f"Processed {df.count()} rows on spot instances")

spark.stop()
client.delete_session(name=session_name_3)
print(f"Session {session_name_3} stopped and cleaned up.")

## 5. Custom Session Naming

Override default auto-generated names (`spark-connect-<uuid>`) with an explicit session identifier.

In [ ]:
session_name = f"custom-session-{uuid.uuid4().hex[:8]}"

spark = client.connect(
    num_executors=1,
    resources_per_executor={"cpu": "1", "memory": "512m"},
    spark_conf={"spark.serializer": "org.apache.spark.serializer.KryoSerializer"},
    options=[
        Name(session_name),
        Labels({"app": "spark", "team": "data-eng"}),
    ],
)

print(f"Spark session created with custom name: {session_name}")
df = spark.range(100)
print(f"Processed {df.count()} rows")

spark.stop()
client.delete_session(name=session_name)
print(f"Session {session_name} stopped and cleaned up.")

## 6. Combined Production Deployment Pattern

Compose `Driver`, `Executor`, and KEP-107 options into a full production configuration.

In [ ]:
prod_session_name = f"prod-etl-{uuid.uuid4().hex[:8]}"

spark = client.connect(
    driver=Driver(
        resources={"cpu": "1", "memory": "512m"},
    ),
    executor=Executor(
        num_instances=1,
        resources_per_executor={"cpu": "1", "memory": "512m"},
    ),
    spark_conf={
        "spark.app.name": "prod-etl-pipeline",
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.adaptive.coalescePartitions.enabled": "true",
        "spark.dynamicAllocation.enabled": "false",
        "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    },
    options=[
        Name(prod_session_name),
        Labels(
            {
                "app": "etl-pipeline",
                "team": "data-platform",
                "environment": "production",
                "cost-center": "analytics",
                "version": "v2.1.0",
            }
        ),
        Annotations(
            {
                "description": "Production ETL pipeline for customer analytics",
                "owner": "data-platform@company.com",
                "slack-channel": "#data-platform-alerts",
                "pagerduty-service": "spark-prod",
                "runbook": "https://wiki.company.com/spark-etl",
            }
        ),
        NodeSelector(
            {
                "kubernetes.io/os": "linux",
            }
        ),
        Toleration(
            key="spot-instance",
            operator="Exists",
            effect="NoSchedule",
        ),
    ],
)

print(f"Production Spark cluster session established: {prod_session_name}")
df = spark.range(100000)
print(f"Processing {df.count()} rows in production environment")

spark.stop()
client.delete_session(name=prod_session_name)
print(f"Production session {prod_session_name} stopped and cleaned up.")